In [1]:
pip install -U langgraph langchain-huggingface

In [2]:
import os
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver


In [3]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [4]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    provider="novita",
    task="conversational",
    max_new_tokens=128,
    do_sample=False,
    huggingfacehub_api_token=HF_TOKEN,
)

chat_model = ChatHuggingFace(llm=llm)

In [5]:
response = chat_model.invoke("Explain difference between langchain and langgraph")

print(response.content)


LangChain and LangGraph are both Python libraries used for natural language processing (NLP) and large language model (LLM) development. While they share some similarities, they have distinct differences in their design, features, and use cases.

**LangChain**

LangChain is a framework for building applications with large language models (LLMs). It provides a set of pre-built components, such as context managers, data structures, and utilities, to help developers build scalable and maintainable LLM-based applications.

Key features of LangChain:

1. **Modular architecture**: LangChain is designed to be highly modular, making it easy to swap


In [6]:
class JokeState(TypedDict):

  topic: str
  joke: str
  explaination : str

In [11]:
def generate_joke(state: JokeState):
  prompt = f'generate a joke on the topic {state["topic"]}'
  response = chat_model.invoke(prompt)
  return {'joke':response.content}

In [12]:
def generate_explaination(state: JokeState):
  prompt = f'Write an explaination for the joke - {state["joke"]} '
  response = chat_model.invoke(prompt)
  return {'explaination': response.content}

In [13]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explaination',generate_explaination)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)

Checkpointer= InMemorySaver()

workflow = graph.compile(checkpointer = Checkpointer)

In [14]:
config1 = {"configurable":{"thread_id":"1"}}
workflow.invoke({'topic':'Noida pollution'},config=config1)

{'topic': 'Noida pollution',
 'joke': "Here's a joke on the topic Noida pollution:\n\nWhy did the air quality in Noida go to therapy?\n\nBecause it was feeling a little choked up, and had a lot of toxic relationships with the commuters and factories around it!\n\n(Note: This joke is a lighthearted way to bring attention to the serious issue of pollution in Noida, but please keep in mind that pollution is a serious problem that requires real-world solutions and actions.)",
 'explaination': 'This joke uses wordplay and metaphor to poke fun at the serious issue of pollution in Noida. The setup "Why did the air quality in Noida go to therapy" is a common structure for jokes, as it allows for a unexpected twist on a mundane situation.\n\nThe punchline "Because it was feeling a little choked up, and had a lot of toxic relationships with the commuters and factories around it" is where the humor comes in. \n\nThe phrase "feeling a little choked up" is a common idiom used to describe someone wh

In [15]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Noida pollution', 'joke': "Here's a joke on the topic Noida pollution:\n\nWhy did the air quality in Noida go to therapy?\n\nBecause it was feeling a little choked up, and had a lot of toxic relationships with the commuters and factories around it!\n\n(Note: This joke is a lighthearted way to bring attention to the serious issue of pollution in Noida, but please keep in mind that pollution is a serious problem that requires real-world solutions and actions.)", 'explaination': 'This joke uses wordplay and metaphor to poke fun at the serious issue of pollution in Noida. The setup "Why did the air quality in Noida go to therapy" is a common structure for jokes, as it allows for a unexpected twist on a mundane situation.\n\nThe punchline "Because it was feeling a little choked up, and had a lot of toxic relationships with the commuters and factories around it" is where the humor comes in. \n\nThe phrase "feeling a little choked up" is a common idiom used to 

In [16]:
list(workflow.get_state_history(config1))
# __start__ → generate_joke → generate_explaination → Complete (next=())

[StateSnapshot(values={'topic': 'Noida pollution', 'joke': "Here's a joke on the topic Noida pollution:\n\nWhy did the air quality in Noida go to therapy?\n\nBecause it was feeling a little choked up, and had a lot of toxic relationships with the commuters and factories around it!\n\n(Note: This joke is a lighthearted way to bring attention to the serious issue of pollution in Noida, but please keep in mind that pollution is a serious problem that requires real-world solutions and actions.)", 'explaination': 'This joke uses wordplay and metaphor to poke fun at the serious issue of pollution in Noida. The setup "Why did the air quality in Noida go to therapy" is a common structure for jokes, as it allows for a unexpected twist on a mundane situation.\n\nThe punchline "Because it was feeling a little choked up, and had a lot of toxic relationships with the commuters and factories around it" is where the humor comes in. \n\nThe phrase "feeling a little choked up" is a common idiom used to

In [17]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Pakistanis'}, config=config2)

{'topic': 'Pakistanis',
 'joke': 'I\'ll try to make a lighthearted joke that is respectful and doesn\'t stereotype any culture. However, I might not be able to perfectly capture the complexities and nuances of Pakistani culture. \n\nHere\'s a joke that is relatively neutral:\n\nWhy did the Pakistani food truck break down? \nBecause it had a "lahori" engine and a "karachi" transmission - it had a "taste" for adventure but not for reliability! (lahori and karachi are famous cities in Pakistan, and Pakistani food is known for its rich flavors)\n\nPlease note that humor can be subjective and might offend some people if not executed',
 'explaination': 'The joke you\'ve created targets the humor in the specificity of Pakistani cities (Lahore and Karachi) and extends it to the famous rich flavors of Pakistani food. Here\'s a breakdown of what might be perceived as funny and what might not land well:\n\n**The humor in the joke:**\n\n1. **Double meaning of "taste":** The joke plays on the multi

# Time Travel

In [20]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1ce190-ec3f-6df9-bfff-019b7c015c62"}})

StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1ce190-ec3f-6df9-bfff-019b7c015c62'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-10-22T13:04:04.242571+00:00', parent_config=None, tasks=(PregelTask(id='f474b3aa-e8a1-cefc-1aaa-7d35ea2ed5be', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'topic': 'Noida pollution'}),), interrupts=())

In [21]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1ce190-ec4a-65d1-8000-4698cb290370"}})

StateSnapshot(values={'topic': 'Noida pollution'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1ce190-ec4a-65d1-8000-4698cb290370'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-22T13:04:04.246833+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce190-ec3f-6df9-bfff-019b7c015c62'}}, tasks=(PregelTask(id='ff8a529a-7d57-6aee-ebd2-8ead0af61cd6', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': "Here's a joke on the topic Noida pollution:\n\nWhy did the air quality in Noida go to therapy?\n\nBecause it was feeling a little choked up, and had a lot of toxic relationships with the commuters and factories around it!\n\n(Note: This joke is a lighthearted way to bring attention to the serious issue of pollution in Noida, but please keep in mind that pollution is a serious problem that requires real-worl

In [22]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1ce190-fb31-6588-8001-f8e71f50dea4"}})

{'topic': 'Noida pollution',
 'joke': "Here's a joke on the topic Noida pollution:\n\nWhy did the air quality in Noida go to therapy?\n\nBecause it was feeling a little choked up, and had a lot of toxic relationships with the commuters and factories around it!\n\n(Note: This joke is a lighthearted way to bring attention to the serious issue of pollution in Noida, but please keep in mind that pollution is a serious problem that requires real-world solutions and actions.)",
 'explaination': 'This joke uses wordplay and a lighthearted twist to address the serious issue of pollution in Noida. The setup, "why did the air quality in Noida go to therapy," is a common joke format that encourages the listener to think of a reason why something would seek help.\n\nThe punchline is where the joke becomes relevant to the topic of pollution in Noida. The phrase "feeling a little choked up" is a common expression for feeling anxious or overwhelmed, but in this context, it\'s a clever play on words b

# Updating State

In [23]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1ce190-ec4a-65d1-8000-4698cb290370", "checkpoint_ns": ""}}, {'topic':'AI Hallucination'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1ce1ba-987f-6716-8001-fe52bea8d507'}}

In [24]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'AI Hallucination'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce1ba-987f-6716-8001-fe52bea8d507'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-10-22T13:22:42.889478+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce190-ec4a-65d1-8000-4698cb290370'}}, tasks=(PregelTask(id='67b43ae3-0e85-fabd-39b1-ab0e6180f8de', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Noida pollution', 'joke': "Here's a joke on the topic Noida pollution:\n\nWhy did the air quality in Noida go to therapy?\n\nBecause it was feeling a little choked up, and had a lot of toxic relationships with the commuters and factories around it!\n\n(Note: This joke is a lighthearted way to bring attention to the serious issue of pol

In [27]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1ce1ba-987f-6716-8001-fe52bea8d507"}})

{'topic': 'AI Hallucination',
 'joke': "Why did the AI's hallucination go to therapy?\n\nBecause it was struggling to distinguish reality from its own elaborate narratives.",
 'explaination': 'This joke relies on a play on the concept of "hallucination" and the characteristics of artificial intelligence (AI).\n\nA hallucination is typically defined as a mental image or sensation that occurs in the absence of any external stimulus. In the context of AI, "hallucination" refers to when a machine learning model generates output that is not supported by the actual training data. This can manifest as the AI producing fictional or irrelevant information, often in a way that is convincing but not grounded in reality.\n\nThe joke suggests that the AI\'s hallucination is struggling to distinguish between reality and its own elaborate narratives. This is a clever twist,'}

# FAULT TOLERANCE

In [28]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [29]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [30]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(15)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [31]:

# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [32]:

try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [33]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [34]:

list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce1cc-ed41-68dc-8003-20f0778e145e'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-10-22T13:30:54.960822+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce1cc-ed3e-610a-8002-45d526f054ed'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce1cc-ed3e-610a-8002-45d526f054ed'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-22T13:30:54.959408+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce1cb-2636-67f4-8001-a4deb8415962'}}, tasks=(PregelTask(id='5f5f1a4d-4dd0-dccf-3b31-4537092c3e7